# Análise · Quatro bairros com queda fora da geração distribuída
### V-Energy Grid 360 — diagnóstico da receita


Dá o **notebook de origem** às afirmações da seção 5.5 do `docs/diagnostico_receita.md`. Só lê tabelas já existentes (`gold.fato_consumo_faturamento`, `gold.dim_uc`, `gold.receita_decomposicao`) e **não grava nada**.

**Pergunta do Henrique:** "se foi volume, quero saber onde e por quê."

**Caminho:**
1. A `receita_decomposicao` mostra que **quatro bairros** (Iputinga e Macaxeira, em Recife; Jardim Jordão e Prazeres, em Jaboatão) respondem por 31,5% da perda de volume residencial.
2. Separando a perda das UCs **com** e **sem** GD, em 11 dos 15 bairros que mais perdem a GD explica tudo, e nesses quatro **63% da perda não vem da GD**.
3. A série mensal mostra uma **mudança de patamar** de cerca de −19%, entre maio e junho de 2025.
4. A distribuição entre as UCs mostra que a queda é **concentrada**: 21,7% das UCs sem GD desses bairros caíram 40% ou mais, contra 0,02% nos demais.
5. A causa **não está nos dados disponíveis**. Sem GD, sem troca de medidor, sem mudança no tipo de leitura e sem alimentador comum. Perdas não técnicas é uma **hipótese a validar por inspeção**.

**Cuidados:** é associação. O limiar de 40% é uma escolha nossa. A estimativa de energia usa a média mensal dos meses comparáveis (sem novembro e fevereiro) e multiplica por 12.

In [0]:
from pyspark.sql import functions as F

catalog = "v_energy"
gold_schema = f"{catalog}.gold"

# Os quatro bairros que concentram a queda sem GD (município, bairro)
QUATRO = [("Recife", "Iputinga"), ("Jaboatão dos Guararapes", "Jardim Jordão"),
          ("Recife", "Macaxeira"), ("Jaboatão dos Guararapes", "Prazeres")]
cond = F.lit(False)
for m, b in QUATRO:
    cond = cond | ((F.col("municipio") == m) & (F.col("bairro") == b))
grupo = F.when(cond, "quatro_bairros").otherwise("demais")

fato = spark.table(f"{gold_schema}.fato_consumo_faturamento").filter("classe = 'RESIDENCIAL'").withColumn("grupo", grupo)
dim = spark.table(f"{gold_schema}.dim_uc").filter("classe = 'RESIDENCIAL'")
rec = spark.table(f"{gold_schema}.receita_decomposicao").filter("classe = 'RESIDENCIAL'")
print("residencial na fato:", f"{fato.count():,}".replace(",", "."))

residencial na fato: 1.833.494


## 1. Os bairros que mais perderam volume residencial, com e sem GD

`perda_ucs_gd` é a variação do volume das UCs que têm GD no fim do período 1; `perda_sem_gd` é o resto. Esperado: nos quatro primeiros, `pct_perda_sem_gd` entre **−12,4% e −17,3%**; nos demais do topo, **positivo, de +3,6% a +4,6%**.

In [0]:
res = (
    rec.withColumn("perda_kwh", F.col("volume_kwh_1") - F.col("volume_kwh_0"))
    .withColumn("perda_ucs_gd", F.col("volume_kwh_ucs_com_gd_1") - F.col("volume_kwh_ucs_com_gd_0"))
    .withColumn("perda_sem_gd", F.col("perda_kwh") - F.col("perda_ucs_gd"))
    .withColumn("pct_perda_sem_gd", F.round(F.expr("try_divide(perda_sem_gd, volume_kwh_0 - volume_kwh_ucs_com_gd_0)") * 100, 1))
    .withColumn("pct_ucs_com_gd", F.round(F.expr("try_divide(qtd_ucs_com_gd_1, qtd_ucs_periodo_1)") * 100, 1))
)
display(res.select("municipio", "bairro", "perda_kwh", "perda_ucs_gd", "perda_sem_gd", "pct_perda_sem_gd",
                   "pct_ucs_com_gd", "qtd_trocas_medidor_1").orderBy("perda_kwh").limit(15))

# Quanto os quatro representam da perda residencial (esperado: 1.381.320 de 4.379.684 kWh = 31,5%; 62,6% sem GD)
tot = res.agg(F.sum("perda_kwh").alias("perda_total")).first()["perda_total"]
q = res.filter(cond).agg(F.sum("perda_kwh").alias("perda"), F.sum("perda_sem_gd").alias("sem_gd")).first()
print(f"perda residencial total: {tot} | quatro bairros: {q['perda']} ({q['perda'] / tot * 100:.1f}%) | sem GD: {q['sem_gd']} ({q['sem_gd'] / q['perda'] * 100:.1f}%)")

municipio,bairro,perda_kwh,perda_ucs_gd,perda_sem_gd,pct_perda_sem_gd,pct_ucs_com_gd,qtd_trocas_medidor_1
Recife,Iputinga,-496099.00,-197649.00,-298450.00,-15.6,9.2,113
Jaboatão dos Guararapes,Jardim Jordão,-439331.00,-163415.00,-275916.00,-15.7,8.0,89
Recife,Macaxeira,-232678.00,-69401.00,-163277.00,-17.3,6.7,54
Jaboatão dos Guararapes,Prazeres,-213212.00,-86170.00,-127042.00,-12.4,7.6,71
Recife,Brasília Teimosa,-161506.00,-298948.00,137442.00,4.1,8.1,249
Jaboatão dos Guararapes,Barra de Jangada,-76744.00,-124033.00,47289.00,4.5,10.1,84
São Lourenço da Mata,Pixete,-68433.00,-117515.00,49082.00,4.5,9.1,82
Paulista,Fragoso,-67489.00,-133017.00,65528.00,4.6,8.3,93
Igarassu,Saramandaia,-62795.00,-89014.00,26219.00,3.6,11.2,70
Jaboatão dos Guararapes,Sucupira,-61336.00,-101887.00,40551.00,4.1,7.7,75


perda residencial total: -4379684.00 | quatro bairros: -1381320.00 (31.5%) | sem GD: -864685.00 (62.6%)


## 2. A série mensal: uma mudança de patamar

Consumo médio por UC e mês, **residencial, só meses sem GD**. Esperado: a razão quatro bairros ÷ demais fica entre **1,005 e 1,011** de 2022-07 a 2025-04, cai para **0,914** em 2025-05 e fica entre **0,813 e 0,822** de 2025-06 a 2026-06 (consumo médio por UC no período 0 → período 1: demais 178,7 → 176,8 kWh, quatro bairros 180,2 → 144,4).

In [0]:
serie = (fato.filter("NOT possui_gd_no_mes")
         .groupBy("mes_referencia").pivot("grupo", ["demais", "quatro_bairros"])
         .agg(F.round(F.avg("consumo_kwh"), 1))
         .withColumn("razao", F.round(F.col("quatro_bairros") / F.col("demais"), 3))
         .orderBy("mes_referencia"))
serie.show(60)

+--------------+------+--------------+-----+
|mes_referencia|demais|quatro_bairros|razao|
+--------------+------+--------------+-----+
|       2022-07| 100.7|         101.8|1.011|
|       2022-08| 101.5|         102.3|1.008|
|       2022-09| 123.5|         124.6|1.009|
|       2022-10| 169.9|         171.4|1.009|
|       2022-11| 210.7|         212.1|1.007|
|       2022-12| 225.9|         227.9|1.009|
|       2023-01| 240.6|         243.0| 1.01|
|       2023-02| 229.1|         231.0|1.008|
|       2023-03| 232.4|         234.4|1.009|
|       2023-04| 211.4|         212.8|1.007|
|       2023-05| 171.0|         172.7| 1.01|
|       2023-06| 127.3|         128.5|1.009|
|       2023-07| 100.7|         101.4|1.007|
|       2023-08| 101.5|         102.1|1.006|
|       2023-09| 123.5|         124.4|1.007|
|       2023-10| 169.8|         170.9|1.006|
|       2023-11| 210.5|         212.0|1.007|
|       2023-12| 225.7|         227.4|1.008|
|       2024-01| 240.0|         241.8|1.008|
|       20

## 3. A queda é concentrada, e não uniforme

Variação contra o mesmo mês do ano anterior, UCs sem GD, nos meses sem o salto de novembro e fevereiro. Esperado: **demais** `[-12,0; -6,5; 0,0; 7,0; 13,6]` e **quatro bairros** `[-56,0; -15,7; -3,5; 4,7; 12,1]` (p10, p25, p50, p75, p90). Só a ponta de baixo difere: o p90 é parecido (+12,1% e +13,6%), então não é uma queda de todas as UCs.

In [0]:
a = fato.filter("NOT possui_gd_no_mes AND consumo_kwh_ano_anterior IS NOT NULL "
                "AND mes_referencia IN ('2025-12','2026-01','2026-03','2026-04')")
(a.groupBy("grupo").agg(F.count("*").alias("linhas"),
     F.expr("percentile_approx(cast(variacao_consumo_ano_anterior_pct as double), array(0.1,0.25,0.5,0.75,0.9))").alias("p10_p25_p50_p75_p90"))
  .show(truncate=False))

+--------------+------+------------------------------------+
|grupo         |linhas|p10_p25_p50_p75_p90                 |
+--------------+------+------------------------------------+
|demais        |130159|[-12.0, -6.47, 0.0, 6.96, 13.61]    |
|quatro_bairros|11155 |[-56.03, -15.69, -3.52, 4.66, 12.12]|
+--------------+------+------------------------------------+



## 4. O que a queda não é

**Tipo de leitura e correção de consumo.** Esperado: a parcela de leituras não reais fica estável de 2025-03 a 2025-08 (~23% nos quatro bairros, ~18% nos demais, **já antes do degrau**), e o consumo corrigido é ~0,1% nos dois grupos.

**Alimentadores.** Esperado: seis alimentadores (`ALM-REC-024`, `ALM-JAB-007`, `ALM-REC-027`, `ALM-REC-028`, `ALM-JAB-013` e `ALM-JAB-014`), nenhum comum aos quatro bairros e nenhum dos três do evento de julho de 2025 (`ALM-REC-008`, `ALM-SAO-004` e `ALM-JAB-004`).

**Troca de medidor.** As trocas ocorrem de setembro de 2025 a março de 2026, depois do degrau (a seção 1 mostra ~10,8 trocas por 100 UCs nos quatro bairros, contra ~14,2 nos demais do topo).

In [0]:
(fato.filter("mes_referencia BETWEEN '2025-03' AND '2025-08'")
     .groupBy("mes_referencia", "grupo")
     .agg(F.round(F.avg((F.col("tipo_leitura") != "REAL").cast("int")) * 100, 1).alias("pct_leitura_nao_real"),
          F.round(F.avg(F.col("consumo_corrigido").cast("int")) * 100, 1).alias("pct_consumo_corrigido"))
     .orderBy("mes_referencia", "grupo").show(20))

(dim.filter(cond).groupBy("municipio", "bairro", "alimentador").count()
    .orderBy("municipio", "bairro", F.desc("count")).show(40, truncate=False))

+--------------+--------------+--------------------+---------------------+
|mes_referencia|         grupo|pct_leitura_nao_real|pct_consumo_corrigido|
+--------------+--------------+--------------------+---------------------+
|       2025-03|        demais|                18.6|                  0.1|
|       2025-03|quatro_bairros|                24.5|                  0.1|
|       2025-04|        demais|                18.3|                  0.1|
|       2025-04|quatro_bairros|                22.8|                  0.0|
|       2025-05|        demais|                18.6|                  0.1|
|       2025-05|quatro_bairros|                23.0|                  0.1|
|       2025-06|        demais|                18.2|                  0.1|
|       2025-06|quatro_bairros|                22.7|                  0.1|
|       2025-07|        demais|                18.4|                  0.1|
|       2025-07|quatro_bairros|                23.4|                  0.1|
|       2025-08|        d

## 5. Quantas UCs e quanta energia

Compara cada UC **com ela mesma**: média do consumo dos meses de `2024-07` a `2025-04` (sem novembro e fevereiro) contra os mesmos meses de `2025-07` a `2026-04`. Conta as UCs **sem GD** que caíram **40% ou mais**.

Esperado: **demais** 32.370 UCs, 7 com queda forte (0,02%); **quatro bairros** 2.774 UCs, **601 (21,7%)**, mediana da queda das afetadas de **−55,1%**, e **1.059.594 kWh por ano** (média mensal da diferença × 12).

A 40% é uma escolha nossa. O grupo `demais` mostra o ruído de fundo: 7 UCs em 32 mil.

In [0]:
meses_depois = ['2025-07', '2025-08', '2025-09', '2025-10', '2025-12', '2026-01', '2026-03', '2026-04']
meses_antes  = ['2024-07', '2024-08', '2024-09', '2024-10', '2024-12', '2025-01', '2025-03', '2025-04']
ucs_sem_gd = dim.filter("NOT possui_gd").select("id_uc")

r = (fato.join(ucs_sem_gd, "id_uc", "left_semi")
     .withColumn("periodo", F.when(F.col("mes_referencia").isin(meses_antes), "antes")
                             .when(F.col("mes_referencia").isin(meses_depois), "depois"))
     .filter("periodo IS NOT NULL")
     .groupBy("id_uc", "grupo", "periodo").agg(F.avg("consumo_kwh").alias("media"))
     .groupBy("id_uc", "grupo").pivot("periodo", ["antes", "depois"]).agg(F.first("media"))
     .filter("antes > 0 AND depois IS NOT NULL")
     .withColumn("queda_pct", (F.col("depois") / F.col("antes") - 1) * 100)
     .withColumn("queda_forte", F.col("queda_pct") <= -40))

(r.groupBy("grupo").agg(
    F.count("*").alias("ucs"),
    F.sum(F.col("queda_forte").cast("int")).alias("ucs_queda_40"),
    F.round(F.avg(F.col("queda_forte").cast("int")) * 100, 1).alias("pct_ucs_queda_40"),
    F.expr("percentile_approx(CASE WHEN queda_forte THEN queda_pct END, 0.5)").alias("mediana_queda_das_afetadas"),
    F.round(F.sum(F.when(F.col("queda_forte"), F.col("antes") - F.col("depois"))) * 12, 0).alias("kwh_ano_perdidos"))
  .show())

+--------------+-----+------------+----------------+--------------------------+----------------+
|         grupo|  ucs|ucs_queda_40|pct_ucs_queda_40|mediana_queda_das_afetadas|kwh_ano_perdidos|
+--------------+-----+------------+----------------+--------------------------+----------------+
|        demais|32370|           7|             0.0|         -41.4387031408308|          8429.0|
|quatro_bairros| 2774|         601|            21.7|       -55.112881806108895|       1059594.0|
+--------------+-----+------------+----------------+--------------------------+----------------+

